# 🏠 House Price Prediction System
### AIML Summer Internship 2026 — IIHMF, MNNIT Allahabad
**Dataset:** Kaggle — House Prices: Advanced Regression Techniques

## Phase 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import pickle

print('✅ All libraries imported successfully!')

## Phase 2: Load Dataset

In [ ]:
# Download from Kaggle: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
# Place train.csv in the Dataset/ folder

df = pd.read_csv('../Dataset/train.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

In [ ]:
# Basic Info
print('=== Dataset Info ===')
df.info()

In [ ]:
# Statistical Summary
print('=== Statistical Summary ===')
df.describe()

## Phase 3: Data Preprocessing

In [ ]:
# Check Missing Values
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_percent})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)
print(f'Columns with missing values: {len(missing_df)}')
missing_df

In [ ]:
# Drop columns with more than 45% missing values
cols_to_drop = missing_df[missing_df['Percentage'] > 45].index.tolist()
print(f'Dropping columns: {cols_to_drop}')
df.drop(columns=cols_to_drop, inplace=True)

# Fill numerical missing values with median
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical missing values with mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Missing values after treatment: {df.isnull().sum().sum()}')
print(f'Dataset shape after cleaning: {df.shape}')

In [ ]:
# Remove Duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Shape after removing duplicates: {df.shape}')

In [ ]:
# Outlier Treatment on SalePrice using IQR
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['SalePrice'] < lower) | (df['SalePrice'] > upper)]
print(f'Outliers in SalePrice: {len(outliers)}')
df = df[(df['SalePrice'] >= lower) & (df['SalePrice'] <= upper)]
print(f'Shape after outlier removal: {df.shape}')

## Phase 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice Distribution', fontsize=14)
axes[0].set_xlabel('Sale Price')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log(SalePrice) Distribution', fontsize=14)
axes[1].set_xlabel('Log Sale Price')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../Documentation/saleprice_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Target variable distribution plotted')

In [ ]:
# Correlation Heatmap (Top Numerical Features)
num_df = df.select_dtypes(include=[np.number])
corr = num_df.corr()['SalePrice'].abs().sort_values(ascending=False)
top_features = corr[1:16].index.tolist()

plt.figure(figsize=(12, 9))
sns.heatmap(num_df[top_features + ['SalePrice']].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Top Features vs SalePrice', fontsize=14)
plt.tight_layout()
plt.savefig('../Documentation/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Correlation heatmap plotted')

In [ ]:
# Scatter Plots — Top 4 features vs SalePrice
scatter_features = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', 'OverallQual']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(scatter_features):
    if feat in df.columns:
        axes[i].scatter(df[feat], df['SalePrice'], alpha=0.4, color='steelblue', s=15)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('SalePrice')
        axes[i].set_title(f'{feat} vs SalePrice')

plt.tight_layout()
plt.savefig('../Documentation/scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Scatter plots plotted')

In [ ]:
# Boxplot — SalePrice by Overall Quality
plt.figure(figsize=(12, 6))
sns.boxplot(x='OverallQual', y='SalePrice', data=df, palette='Blues')
plt.title('SalePrice by Overall Quality', fontsize=14)
plt.xlabel('Overall Quality (1-10)')
plt.ylabel('Sale Price')
plt.tight_layout()
plt.savefig('../Documentation/boxplot_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Boxplot plotted')

## Phase 5: Feature Engineering

In [ ]:
# Create New Features
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasPool'] = (df['PoolArea'] > 0).astype(int) if 'PoolArea' in df.columns else 0

print('✅ New features created:')
print(' - HouseAge, RemodAge, TotalSF, TotalBath, HasGarage, HasPool')

In [ ]:
# Encode Categorical Columns
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f'✅ Encoded {len(cat_cols)} categorical columns')
print(f'Dataset shape: {df.shape}')

In [ ]:
# Select Top Features using Correlation
corr_with_target = df.corr()['SalePrice'].abs().sort_values(ascending=False)
top_n = 20
selected_features = corr_with_target[1:top_n+1].index.tolist()
print(f'✅ Top {top_n} selected features:')
print(selected_features)

## Phase 6: Model Building

In [ ]:
# Prepare X and y
X = df[selected_features]
y = df['SalePrice']

# Log transform target for better results
y_log = np.log1p(y)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✅ Train size: {X_train.shape}, Test size: {X_test.shape}')

In [ ]:
# ---- Model 1: Linear Regression ----
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print('=== Model 1: Linear Regression ===')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_lr):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}')
print(f'R²   : {r2_score(y_test, y_pred_lr):.4f}')

In [ ]:
# ---- Model 2: Random Forest Regressor ----
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Model 2: Random Forest Regressor ===')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_rf):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}')
print(f'R²   : {r2_score(y_test, y_pred_rf):.4f}')

In [ ]:
# ---- Model 3: XGBoost Regressor ----
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print('=== Model 3: XGBoost Regressor ===')
print(f'MAE  : {mean_absolute_error(y_test, y_pred_xgb):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.4f}')
print(f'R²   : {r2_score(y_test, y_pred_xgb):.4f}')

## Phase 7: Model Comparison & Evaluation

In [ ]:
# Comparison Table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_xgb)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    ],
    'R2_Score': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_xgb)
    ]
})
results = results.round(4)
print('=== Model Comparison ===')
print(results.to_string(index=False))

In [ ]:
# Model Comparison Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['MAE', 'RMSE', 'R2_Score']
colors = ['#e74c3c', '#f39c12', '#2ecc71']

for i, metric in enumerate(metrics):
    axes[i].bar(results['Model'], results[metric], color=colors[i], edgecolor='white', width=0.5)
    axes[i].set_title(f'{metric} Comparison', fontsize=12)
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Model Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../Documentation/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved')

In [ ]:
# Feature Importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=selected_features)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 7))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../Documentation/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved')

## Phase 8: Save Best Model

In [ ]:
# Save Best Model (XGBoost) + Scaler + Feature List
best_model = xgb  # Change if Random Forest wins

with open('../Model/xgboost_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('../Model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../Model/features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)

print('✅ Model saved: Model/xgboost_model.pkl')
print('✅ Scaler saved: Model/scaler.pkl')
print('✅ Features saved: Model/features.pkl')
print('\n🎉 Project Complete! Now run the Streamlit app.')